# GENESIS Training Smoke Test

Validates the full training pipeline on GPU before committing to Lambda.

**What this tests:**
- HLRT model instantiation (496M params)
- MuonAdamWHybrid optimizer (Muon 2D + AdamW 1D routing)
- WSD scheduler (warmup-stable-decay)
- BootstrapTrainer with TrainingOrchestrator
- Next-token prediction loss converges
- Gradient flow through all 3 tiers
- KV cache inference after training

**Requirements:** Colab Pro with A100 or V100 GPU

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Clone repo and install
!git clone https://github.com/brendenk6/LLM-Research-.git genesis
%cd genesis
!pip install -e ".[data,training]" -q
!pip install tiktoken -q  # ensure tiktoken is available

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

import math
import time
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from genesis.model.hlrt import HLRT, HLRTConfig
from genesis.training.train_phase1_bootstrap import BootstrapTrainer
from olympus.optim.muon_adamw_hybrid import MuonAdamWHybrid
from olympus.optim.schedulers import WSDScheduler
from olympus.data.tokenizer import TokenizerWrapper

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Tokenizer + Synthetic Data

We use tiktoken (cl100k_base) and generate synthetic training data.
Real data lives in `cleaned/` JSONL files — we'll use those on Lambda.
For smoke testing, synthetic data is enough to verify loss decreases.

In [ ]:
tokenizer = TokenizerWrapper(backend="tiktoken")
print(f"Backend: {tokenizer.backend_name}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Special tokens: {tokenizer.special_token_ids()}")

In [ ]:
class SyntheticNTPDataset(Dataset):
    """Synthetic dataset for smoke-testing next-token prediction.

    Generates repeating patterns that any model should be able to memorize,
    proving the training loop works. Each sequence is a repeated pattern
    of random tokens — if loss doesn't drop, something is broken.
    """

    def __init__(self, num_samples: int, seq_len: int, vocab_size: int, seed: int = 42):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size

        rng = torch.Generator().manual_seed(seed)
        # Create short patterns (8-16 tokens) and tile them to seq_len
        # This makes the data learnable — model just needs to memorize patterns
        self.data = []
        for _ in range(num_samples):
            pattern_len = torch.randint(8, 17, (1,), generator=rng).item()
            pattern = torch.randint(0, min(vocab_size, 1000), (pattern_len,), generator=rng)
            # Tile pattern to fill seq_len
            repeats = (seq_len // pattern_len) + 1
            seq = pattern.repeat(repeats)[:seq_len]
            self.data.append(seq)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return {"input_ids": self.data[idx]}


# Config: 512 samples, 512 seq_len — small enough for fast iteration
SEQ_LEN = 512
NUM_SAMPLES = 512
BATCH_SIZE = 8

dataset = SyntheticNTPDataset(
    num_samples=NUM_SAMPLES,
    seq_len=SEQ_LEN,
    vocab_size=tokenizer.vocab_size,
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Dataset: {len(dataset)} samples, {SEQ_LEN} tokens each")
print(f"Batches per epoch: {len(dataloader)}")
print(f"Tokens per epoch: {NUM_SAMPLES * SEQ_LEN:,}")

## 3. Model Configuration

Full 496M param HLRT config. If VRAM is tight (V100 16GB), the fallback
cell below scales it down.

In [ ]:
# Detect VRAM and pick model size
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    vram_gb = 0

print(f"Available VRAM: {vram_gb:.1f} GB")

if vram_gb >= 30:  # A100 40GB or better
    print("-> Using FULL 496M config")
    config = HLRTConfig(
        vocab_size=tokenizer.vocab_size,
        d_model=1024,
        tier1_num_layers=12,
        tier1_num_heads=16,
        tier1_num_kv_heads=4,
        tier1_max_seq_len=SEQ_LEN,
        tier1_use_flash_attention=False,  # vanilla attn for compatibility
        tier2_d_model=1536,
        tier2_num_layers=8,
        tier2_num_heads=16,
        tier2_max_seq_len=SEQ_LEN // 4,
        tier2_use_flash_attention=False,
        tier3_d_model=1024,
        tier3_num_layers=5,
        tier3_num_heads=8,
        tier3_max_seq_len=SEQ_LEN // 8,
        tier3_use_flash_attention=False,
        tier3_recurrence_steps=4,
        gate1_chunk_size=32,
        num_latent_vectors=8,
    )
elif vram_gb >= 14:  # V100 16GB or T4
    print("-> Using MEDIUM config (~120M) for limited VRAM")
    config = HLRTConfig(
        vocab_size=tokenizer.vocab_size,
        d_model=512,
        tier1_num_layers=6,
        tier1_num_heads=8,
        tier1_num_kv_heads=2,
        tier1_max_seq_len=SEQ_LEN,
        tier1_use_flash_attention=False,
        tier2_d_model=768,
        tier2_num_layers=4,
        tier2_num_heads=8,
        tier2_max_seq_len=SEQ_LEN // 4,
        tier2_use_flash_attention=False,
        tier3_d_model=512,
        tier3_num_layers=3,
        tier3_num_heads=8,
        tier3_max_seq_len=SEQ_LEN // 8,
        tier3_use_flash_attention=False,
        tier3_recurrence_steps=2,
        gate1_chunk_size=16,
        num_latent_vectors=4,
    )
else:
    print("-> Using TINY config (~15M) for CPU/low VRAM")
    config = HLRTConfig(
        vocab_size=tokenizer.vocab_size,
        d_model=256,
        tier1_num_layers=4,
        tier1_num_heads=4,
        tier1_num_kv_heads=2,
        tier1_max_seq_len=SEQ_LEN,
        tier1_use_flash_attention=False,
        tier2_d_model=384,
        tier2_num_layers=2,
        tier2_num_heads=4,
        tier2_max_seq_len=SEQ_LEN // 4,
        tier2_use_flash_attention=False,
        tier3_d_model=256,
        tier3_num_layers=1,
        tier3_num_heads=4,
        tier3_max_seq_len=SEQ_LEN // 8,
        tier3_use_flash_attention=False,
        tier3_recurrence_steps=2,
        gate1_chunk_size=8,
        num_latent_vectors=2,
    )

In [ ]:
# Instantiate model
model = HLRT(config)

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model parameters: {param_count / 1e6:.1f}M")
print(f"Trainable: {trainable_count / 1e6:.1f}M")
print(f"Estimated weight memory: {param_count * 4 / 1e9:.2f} GB (fp32)")
print(f"Config: d_model={config.d_model}, T1={config.tier1_num_layers}L, T2={config.tier2_num_layers}L, T3={config.tier3_num_layers}L")

## 4. Optimizer + Scheduler

In [ ]:
TOTAL_STEPS = 200  # Just enough to see loss curve
WARMUP_STEPS = 20
GRAD_ACCUM = 2

optimizer = MuonAdamWHybrid(
    model.parameters(),
    lr_muon=0.02,
    lr_adamw=3e-4,
    weight_decay_adamw=0.01,
    ns_steps=5,
)

scheduler = WSDScheduler(
    optimizer,
    base_lr=3e-4,
    min_lr=1e-5,
    warmup_steps=WARMUP_STEPS,
    total_steps=TOTAL_STEPS,
    decay_start=int(TOTAL_STEPS * 0.8),
)

# Count Muon vs AdamW params
muon_params = sum(p.numel() for p in model.parameters() if p.dim() == 2)
adamw_params = sum(p.numel() for p in model.parameters() if p.dim() != 2)
print(f"Muon (2D) params: {muon_params / 1e6:.1f}M")
print(f"AdamW (1D) params: {adamw_params / 1e6:.1f}M")
print(f"Total steps: {TOTAL_STEPS}, warmup: {WARMUP_STEPS}, grad_accum: {GRAD_ACCUM}")

## 5. Training Loop

Uses the full BootstrapTrainer pipeline — same code that will run on Lambda.

In [ ]:
trainer = BootstrapTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    tokenizer=tokenizer,
    config={
        "gradient_accumulation_steps": GRAD_ACCUM,
        "max_grad_norm": 1.0,
        "log_interval": 25,
        "save_interval": 0,  # don't save during smoke test
        "checkpoint_dir": "/content/checkpoints",
        "device": str(device),
    },
)

print("BootstrapTrainer initialized.")
print(f"Model on: {next(model.parameters()).device}")

In [ ]:
# Training loop with detailed logging
losses = []
lrs = []
tok_per_sec = []
tier_activations = {"tier1": 0, "tier2": 0, "tier3": 0}

num_epochs = max(1, TOTAL_STEPS * GRAD_ACCUM // len(dataloader) + 1)
print(f"Running ~{TOTAL_STEPS} optimizer steps ({num_epochs} epochs)...")
print("=" * 70)

train_start = time.time()
step_count = 0

for epoch in range(num_epochs):
    for batch in dataloader:
        metrics = trainer.train_step(batch)
        step_count += 1

        losses.append(metrics["loss"])
        lrs.append(metrics["lr"])
        tok_per_sec.append(metrics["tokens_per_sec"])

        if step_count % 25 == 0:
            avg_recent = sum(losses[-25:]) / len(losses[-25:])
            print(
                f"Step {step_count:4d} | loss={metrics['loss']:.4f} | "
                f"avg25={avg_recent:.4f} | lr={metrics['lr']:.2e} | "
                f"{metrics['tokens_per_sec']:.0f} tok/s"
            )

        if trainer.orchestrator.global_step >= TOTAL_STEPS:
            break
    if trainer.orchestrator.global_step >= TOTAL_STEPS:
        break

elapsed = time.time() - train_start
total_tokens = step_count * BATCH_SIZE * SEQ_LEN

print("=" * 70)
print(f"Training complete: {step_count} micro-steps in {elapsed:.1f}s")
print(f"Global optimizer steps: {trainer.orchestrator.global_step}")
print(f"Total tokens processed: {total_tokens:,}")
print(f"Avg throughput: {total_tokens / elapsed:.0f} tok/s")
print(f"First loss: {losses[0]:.4f}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Loss reduction: {((losses[0] - losses[-1]) / losses[0] * 100):.1f}%")

## 6. Loss Curve

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(losses, alpha=0.3, label="per-step")
# Smoothed loss (window of 10)
window = 10
if len(losses) > window:
    smoothed = [sum(losses[max(0,i-window):i+1]) / len(losses[max(0,i-window):i+1]) for i in range(len(losses))]
    axes[0].plot(smoothed, color="red", linewidth=2, label=f"smoothed (w={window})")
axes[0].set_xlabel("Micro-step")
axes[0].set_ylabel("Loss")
axes[0].set_title("NTP Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LR schedule
axes[1].plot(lrs)
axes[1].set_xlabel("Micro-step")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("WSD Schedule")
axes[1].grid(True, alpha=0.3)

# Throughput
axes[2].plot(tok_per_sec, alpha=0.3)
if len(tok_per_sec) > window:
    smoothed_tps = [sum(tok_per_sec[max(0,i-window):i+1]) / len(tok_per_sec[max(0,i-window):i+1]) for i in range(len(tok_per_sec))]
    axes[2].plot(smoothed_tps, color="green", linewidth=2)
axes[2].set_xlabel("Micro-step")
axes[2].set_ylabel("Tokens/sec")
axes[2].set_title("Throughput")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Validation Checks

In [ ]:
# ---- Check 1: Loss decreased ----
first_10_avg = sum(losses[:10]) / min(len(losses), 10)
last_10_avg = sum(losses[-10:]) / min(len(losses), 10)
loss_dropped = last_10_avg < first_10_avg * 0.9  # at least 10% drop

print("VALIDATION CHECKS")
print("=" * 50)
print(f"[{'PASS' if loss_dropped else 'FAIL'}] Loss decreased: {first_10_avg:.4f} -> {last_10_avg:.4f}")

# ---- Check 2: No NaN/Inf in parameters ----
has_nan = any(torch.isnan(p).any() or torch.isinf(p).any() for p in model.parameters())
print(f"[{'FAIL' if has_nan else 'PASS'}] No NaN/Inf in model weights")

# ---- Check 3: Gradients flowed to all tiers ----
model.train()
test_ids = torch.randint(0, min(tokenizer.vocab_size, 1000), (1, 64), device=device)
out = model(test_ids)
loss = F.cross_entropy(
    out["logits"][:, :-1].reshape(-1, out["logits"].size(-1)),
    test_ids[:, 1:].reshape(-1),
)
loss.backward()

# Check tier1
tier1_grads = any(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in model.tier1.parameters()
)
print(f"[{'PASS' if tier1_grads else 'FAIL'}] Tier 1 gradients flowing")

# Check tier2 (may not activate on short sequences)
tier2_grads = any(
    p.grad is not None and p.grad.abs().sum() > 0
    for p in model.tier2.parameters()
)
print(f"[{'PASS' if tier2_grads else 'WARN'}] Tier 2 gradients flowing (may skip if gate didn't fire)")

# Check embeddings
emb_grads = model.embeddings.weight.grad is not None and model.embeddings.weight.grad.abs().sum() > 0
print(f"[{'PASS' if emb_grads else 'FAIL'}] Embedding gradients flowing")

model.zero_grad(set_to_none=True)

# ---- Check 4: Forward pass produces valid logits ----
model.eval()
with torch.no_grad():
    test_out = model(test_ids)
    logits = test_out["logits"]
    valid_logits = not (torch.isnan(logits).any() or torch.isinf(logits).any())
    print(f"[{'PASS' if valid_logits else 'FAIL'}] Logits are finite")
    print(f"  Logit range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")

# ---- Summary ----
all_pass = loss_dropped and not has_nan and tier1_grads and emb_grads and valid_logits
print("\n" + "=" * 50)
if all_pass:
    print("ALL CHECKS PASSED -- pipeline is ready for Lambda.")
else:
    print("SOME CHECKS FAILED -- investigate before scaling up.")

## 8. Inference Test (KV Cache)

In [ ]:
from genesis.inference.generator import Generator, GenerationConfig
from genesis.inference.sampling import SamplingConfig

model.eval()
gen = Generator(model)

# Generate with KV cache
prompt_ids = torch.randint(0, min(tokenizer.vocab_size, 1000), (1, 16), device=device)
gen_config = GenerationConfig(
    max_new_tokens=32,
    use_kv_cache=True,
    sampling=SamplingConfig(temperature=0.8, top_k=50),
)

result = gen.generate(prompt_ids, gen_config)
print(f"Prompt: {prompt_ids.shape} -> Generated: {result['new_token_ids'].shape}")
print(f"Full sequence: {result['token_ids'].shape}")

stats = result["cascade_stats"]
print(f"\nCascade stats:")
print(f"  Total tokens: {stats.total_tokens}")
print(f"  Tier 1 rate: {stats.tier1_rate:.1%}")
print(f"  Tier 2 rate: {stats.tier2_rate:.1%}")
print(f"  Tier 3 rate: {stats.tier3_rate:.1%}")
print(f"  Compute savings: {stats.compute_savings:.1%}")

# Decode generated tokens
generated_ids = result["new_token_ids"][0].tolist()
decoded = tokenizer.decode(generated_ids)
print(f"\nGenerated text (will be nonsense — model barely trained):")
print(f"  {decoded[:200]}")

## 9. Checkpoint Save/Load Test

In [ ]:
import os

# Save checkpoint
ckpt_path = "/content/checkpoints/smoke_test.pt"
trainer.save_checkpoint(ckpt_path)
ckpt_size = os.path.getsize(ckpt_path) / 1e6
print(f"Checkpoint saved: {ckpt_size:.1f} MB")

# Load into fresh trainer to verify roundtrip
model2 = HLRT(config).to(device)
opt2 = MuonAdamWHybrid(model2.parameters(), lr_muon=0.02, lr_adamw=3e-4)
sched2 = WSDScheduler(opt2, base_lr=3e-4, warmup_steps=WARMUP_STEPS, total_steps=TOTAL_STEPS)

trainer2 = BootstrapTrainer(
    model=model2, optimizer=opt2, scheduler=sched2, tokenizer=tokenizer,
    config={"gradient_accumulation_steps": GRAD_ACCUM, "device": str(device)},
)
trainer2.load_checkpoint(ckpt_path)

# Verify weights match
model.eval()
model2.eval()
with torch.no_grad():
    out1 = model(test_ids)["logits"]
    out2 = model2(test_ids)["logits"]
    match = torch.allclose(out1, out2, atol=1e-5)
    print(f"Checkpoint roundtrip: {'PASS' if match else 'FAIL'}")
    print(f"Restored global_step: {trainer2.orchestrator.global_step}")

del model2, opt2, sched2, trainer2
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 10. Memory + Throughput Profiling

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

    # Run a few training steps to get steady-state memory
    model.train()
    for i, batch in enumerate(dataloader):
        if i >= 5:
            break
        trainer.train_step(batch)

    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    reserved_mem = torch.cuda.max_memory_reserved() / 1e9

    print(f"Peak allocated memory: {peak_mem:.2f} GB")
    print(f"Peak reserved memory: {reserved_mem:.2f} GB")
    print(f"VRAM headroom: {vram_gb - peak_mem:.2f} GB")
    print()

    # Estimate Lambda requirements
    # Full training: batch_size=32, seq_len=2048, grad_accum=8
    scale_factor = (32 / BATCH_SIZE) * (2048 / SEQ_LEN)
    estimated_full = peak_mem * min(scale_factor, 5)  # rough upper bound
    print(f"Estimated full-scale VRAM (batch=32, seq=2048): ~{estimated_full:.0f} GB")
    if estimated_full > 40:
        print("-> Will need gradient checkpointing or smaller batch on A100 40GB")
    elif estimated_full > 24:
        print("-> A100 40GB should work with grad_accum to reduce effective batch")
    else:
        print("-> A100 40GB has plenty of room")
else:
    print("No GPU — skipping memory profiling")

## Summary

If all checks passed, the GENESIS training pipeline is validated:

- HLRT 3-tier architecture initializes and trains correctly
- MuonAdamWHybrid correctly routes 2D->Muon, 1D->AdamW
- WSD scheduler ramps and decays as expected
- BootstrapTrainer + TrainingOrchestrator handle gradient accumulation
- NTP loss decreases (model learns)
- KV cache inference works post-training
- Checkpoints save and load correctly

**Next: Lambda training with real data.**